# NB19: Signals Beyond Niche Breadth

Four analyses to distinguish Tier 1 (resistance) vs Tier 2 (metabolic) gene signals and inform ENIGMA soil microcosm design.

**Blocks:**
1. Community-weighted mean ko/Mb ~ NGSA metal concentration
2. Genus dose-response: Spearman rho (detection freq vs metal quartile medians)
3. Tier 1 vs Tier 2 within-MAG co-occurrence (partial Spearman + permutation p-value)
4. Hotspot genus identity: Tier 1/2 Mann-Whitney U + continuous Spearman
5. Synthesis: experimental design signals table

In [1]:
import os
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import rankdata
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

BASE    = '/home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_metal_ecology'
FIGDIR  = f'{BASE}/data/figures'
os.makedirs(FIGDIR, exist_ok=True)

METALS       = ['Cu', 'Zn', 'Pb', 'Ni', 'Co', 'As', 'Cr', 'Hg']
NGSA_COLS    = {m: f'ngsa_{m}_ppm' for m in METALS}
FOCUS_METALS = ['Cu', 'Cr', 'As', 'Zn']
NGSA_DIST_KM = 200
N_PERM = 1000
N_BOOT = 1000

np.random.seed(42)
print("Setup complete")


Setup complete


## Block 1: Community-Weighted Mean ko/Mb ~ NGSA Metal Concentration

If community-level metal gene investment rises with metal concentration, that confirms community-scale selection (not just genus-level signal).

In [2]:
# ---- Load NGSA data; strip '102.100.100/' prefix to match OTU column IDs ----
ngsa = pd.read_csv(f'{BASE}/data/aus_microbiome/aus_sample_ngsa.csv')
ngsa['Sample_ID'] = ngsa['Sample_ID'].astype(str).str.split('/').str[-1]
ngsa_filt = ngsa[ngsa['ngsa_dist_km'] <= NGSA_DIST_KM].copy()
ngsa_filt = ngsa_filt.set_index('Sample_ID')
print(f"NGSA-filtered samples: {len(ngsa_filt)}")

# ---- Load OTU table and taxonomy ----
print("Loading OTU table...")
otu = pd.read_csv(f'{BASE}/data/aus_microbiome/BASE_16S_OTU.csv.gz', index_col=0)
otu.index   = otu.index.astype(str)
otu.columns = otu.columns.astype(str)
print(f"OTU table: {otu.shape[0]} OTUs x {otu.shape[1]} samples")

tax = pd.read_csv(f'{BASE}/data/aus_microbiome/BASE_16S_taxonomy.csv')
tax['OTUId']       = tax['OTUId'].astype(str)
tax['genus_lower'] = (tax['genus']
                      .str.replace(r'^g__', '', regex=True)
                      .str.strip()
                      .replace({'unclassified': np.nan, '': np.nan})
                      .str.lower())
tax = tax.dropna(subset=['genus_lower'])
genus_map = tax.set_index('OTUId')['genus_lower']   # OTUId -> genus_lower

# ---- Shared samples ----
otu_g  = otu[otu.index.isin(tax['OTUId'])]          # OTUs with genus assignment
shared = sorted(set(otu.columns) & set(ngsa_filt.index))
print(f"OTUs with genus: {len(otu_g):,}  |  Shared samples: {len(shared)}")

otu_sub  = otu_g[shared]                             # counts: OTUs x shared_samples
ng_sub   = ngsa_filt.loc[shared]                     # NGSA metals for shared samples

# ---- Relative abundance per sample ----
col_sums     = otu_sub.sum(axis=0)
otu_rel      = otu_sub.div(col_sums, axis='columns')
otu_rel.index = otu_rel.index.map(genus_map)
otu_rel      = otu_rel[~otu_rel.index.isna()]
g_ra         = otu_rel.groupby(level=0).sum()        # genera x shared_samples (rel abund)

# ---- Presence/absence (also needed by Block 2) ----
otu_sub_m      = otu_sub.copy()
otu_sub_m.index = otu_sub_m.index.map(genus_map)
otu_sub_m      = otu_sub_m[~otu_sub_m.index.isna()]
genus_pres     = otu_sub_m.groupby(level=0).sum() > 0   # genera x samples, bool
print(f"Genera in presence matrix: {len(genus_pres)}")

# ---- Load MAG KO density, aggregate to genus ----
print("Loading MAG KO density...")
mag_ko = pd.read_csv(f'{BASE}/data/mgnify_mag_ko_density.csv')
if mag_ko['genus'].str.startswith('g__').any():
    mag_ko['genus_lower'] = mag_ko['genus'].str[3:].str.lower()
else:
    mag_ko['genus_lower'] = mag_ko['genus'].str.lower()

genus_ko = mag_ko.groupby('genus_lower')[
    ['ko_per_mb_total', 'ko_per_mb_tier1', 'ko_per_mb_tier2']
].mean()
print(f"Genus-level KO density: {len(genus_ko)} genera")

# ---- Compute CWM per sample ----
common = g_ra.index.intersection(genus_ko.index)
print(f"Common genera (OTU intersect MAG KO): {len(common)}")
ra_c = g_ra.loc[common]
ko_c = genus_ko.loc[common]

cwm_records = []
for samp in shared:
    w  = ra_c[samp].values
    ws = w.sum()
    if ws == 0:
        cwm_records.append({'Sample_ID': samp, 'CWM_total': np.nan,
                             'CWM_tier1': np.nan, 'CWM_tier2': np.nan})
    else:
        cwm_records.append({
            'Sample_ID': samp,
            'CWM_total': np.dot(w, ko_c['ko_per_mb_total'].values) / ws,
            'CWM_tier1': np.dot(w, ko_c['ko_per_mb_tier1'].values) / ws,
            'CWM_tier2': np.dot(w, ko_c['ko_per_mb_tier2'].values) / ws,
        })

cwm_df = pd.DataFrame(cwm_records).set_index('Sample_ID')
cwm_df = cwm_df.join(ng_sub[[NGSA_COLS[m] for m in METALS]], how='inner')
print(f"CWM computed for {len(cwm_df)} samples")
print(cwm_df[['CWM_total', 'CWM_tier1', 'CWM_tier2']].describe().round(4))

# ---- Spearman rho: CWM ~ NGSA metal concentration ----
rows = []
for metal in METALS:
    col = NGSA_COLS[metal]
    for cwm_type in ['CWM_total', 'CWM_tier1', 'CWM_tier2']:
        v = cwm_df[[cwm_type, col]].dropna()
        if len(v) < 30:
            continue
        rho, p = stats.spearmanr(v[col], v[cwm_type])
        rows.append({'metal': metal, 'cwm_type': cwm_type, 'rho': rho,
                     'p': p, 'n_samples': len(v)})

cwm_res = pd.DataFrame(rows)
m = len(cwm_res)
ranks = rankdata(cwm_res['p'])
cwm_res['q_BH'] = np.minimum(cwm_res['p'] * m / ranks, 1.0)
cwm_res = cwm_res.sort_values('q_BH')
cwm_res.to_csv(f'{BASE}/data/cwm_ngsa_spearman.csv', index=False)

print("\nAll CWM ~ NGSA Spearman results:")
print(cwm_res[['metal', 'cwm_type', 'rho', 'p', 'q_BH', 'n_samples']].to_string())
sig = cwm_res[cwm_res['q_BH'] < 0.1]
print(f"\nSignificant (q_BH < 0.1): {len(sig)} tests")


NGSA-filtered samples: 1307
Loading OTU table...


OTU table: 91929 OTUs x 1023 samples


OTUs with genus: 18,152  |  Shared samples: 745


Genera in presence matrix: 933
Loading MAG KO density...


Genus-level KO density: 7541 genera
Common genera (OTU intersect MAG KO): 441
CWM computed for 745 samples
       CWM_total  CWM_tier1  CWM_tier2
count   745.0000   745.0000   745.0000
mean      5.0867     2.5863     2.5004
std       0.5191     0.2618     0.3296
min       3.6340     1.8249     1.6775
25%       4.7538     2.4379     2.2772
50%       5.0490     2.5644     2.4833
75%       5.3844     2.7192     2.6726
max       8.2808     4.2766     4.8394

All CWM ~ NGSA Spearman results:
   metal   cwm_type       rho             p      q_BH  n_samples
9     Ni  CWM_total  0.182319  5.940436e-07  0.000007        740
13    Co  CWM_tier1  0.185735  3.298146e-07  0.000008        745
10    Ni  CWM_tier1  0.177443  1.189285e-06  0.000010        740
0     Cu  CWM_total  0.172997  2.540677e-06  0.000015        731
8     Pb  CWM_tier2 -0.167848  4.097118e-06  0.000020        745
22    Hg  CWM_tier1  0.179662  6.529532e-06  0.000026        622
2     Cu  CWM_tier2  0.164457  7.848386e-06  0.000027

## Block 2: Genus Dose-Response Across NGSA Metal Quartiles

Spearman rho between 4 quartile detection frequencies and 4 quartile median concentrations. BH FDR applied across all genus x metal pairs. Note: n=4 limits minimum p to ~0.083; rho is the primary interpretable quantity.

In [3]:
# genus_pres (binary, genera x shared_samples) built in Block 1
print(f"Genus presence matrix: {genus_pres.shape[0]} genera x {genus_pres.shape[1]} samples")

# Note: Spearman with n=4 quartiles has limited resolution (min achievable p~0.083);
# rho values are the primary interpretable output; FDR is applied across all genus-metal pairs.
MIN_Q_SAMPLES = 5
dose_rows = []

for metal in FOCUS_METALS:
    col = NGSA_COLS[metal]
    metal_vals = ng_sub[col].reindex(shared)
    valid_mask = metal_vals.notna()
    valid_samps = metal_vals[valid_mask].index   # Index of valid sample IDs
    valid_metal = metal_vals[valid_mask]          # Series: valid_samps -> metal value

    try:
        q_labels, _ = pd.qcut(valid_metal, q=4, retbins=True, labels=False, duplicates='drop')
    except ValueError:
        print(f"  Skipping {metal}: not enough unique values for quartiles")
        continue

    n_q = q_labels.nunique()
    q_medians = valid_metal.groupby(q_labels).median().values   # median conc per quartile

    pres_sub = genus_pres.loc[:, valid_samps]   # genera x valid_samples

    n_skipped = 0
    for genus in genus_pres.index:
        pres_vec  = pres_sub.loc[genus].values.astype(float)   # aligned with valid_samps
        det_freqs, n_per_q = [], []
        skip = False
        for q in range(n_q):
            qm = (q_labels == q).values   # bool array aligned with valid_samps
            nq = qm.sum()
            if nq < MIN_Q_SAMPLES:
                skip = True
                break
            det_freqs.append(pres_vec[qm].mean())
            n_per_q.append(nq)
        if skip or len(det_freqs) < 3:
            n_skipped += 1
            continue
        rho, p = stats.spearmanr(q_medians[:len(det_freqs)], det_freqs)
        dose_rows.append({'genus_lower': genus, 'metal': metal, 'rho': rho,
                          'p_spearman': p, 'n_quartile_min': min(n_per_q)})
    print(f"  {metal}: {n_skipped} genera skipped (insufficient quartile samples)")

dose_df = pd.DataFrame(dose_rows)
print(f"\nDose-response genus-metal pairs: {len(dose_df)}")

# BH FDR across all tests
m = len(dose_df)
ranks = rankdata(dose_df['p_spearman'])
dose_df['q_BH'] = np.minimum(dose_df['p_spearman'] * m / ranks, 1.0)

# Merge with genus KO density
dose_df = dose_df.merge(
    genus_ko[['ko_per_mb_total', 'ko_per_mb_tier1', 'ko_per_mb_tier2']].reset_index(),
    on='genus_lower', how='left'
)
dose_df.to_csv(f'{BASE}/data/genus_dose_response.csv', index=False)
sig_dose = dose_df[dose_df['q_BH'] < 0.1]
print(f"Significant (q_BH < 0.1): {len(sig_dose)} genus-metal pairs")
if len(sig_dose):
    print(sig_dose.groupby('metal').size().to_dict())

# ---- Spearman rho: per-genus dose-response rho ~ ko_per_mb ----
dr_rows = []
for metal in FOCUS_METALS:
    sub = dose_df[dose_df['metal'] == metal].dropna(
        subset=['rho', 'ko_per_mb_tier1', 'ko_per_mb_tier2'])
    if len(sub) < 20:
        print(f"  {metal}: only {len(sub)} genera with KO data, skipping")
        continue
    for pred in ['ko_per_mb_total', 'ko_per_mb_tier1', 'ko_per_mb_tier2']:
        r, p = stats.spearmanr(sub['rho'], sub[pred])
        dr_rows.append({'metal': metal, 'predictor': pred, 'rho': r, 'p': p,
                        'n_genera': len(sub)})

dr_pgls = pd.DataFrame(dr_rows)
dr_pgls.to_csv(f'{BASE}/data/dose_response_pgls.csv', index=False)
print("\nDose-response rho ~ ko_per_mb Spearman:")
print(dr_pgls.to_string(index=False))

# ---- Figure 10: Cr dose-response rho vs Tier 1 density ----
cr_sub = dose_df[dose_df['metal'] == 'Cr'].dropna(
    subset=['rho', 'ko_per_mb_tier1', 'ko_per_mb_tier2'])
if len(cr_sub) > 0:
    fig, ax = plt.subplots(figsize=(6, 5))
    sc = ax.scatter(cr_sub['ko_per_mb_tier1'], cr_sub['rho'],
                    c=cr_sub['ko_per_mb_tier2'], cmap='viridis', alpha=0.5, s=20, linewidths=0)
    plt.colorbar(sc, ax=ax, label='ko_per_mb_tier2')
    ax.axhline(0, color='grey', lw=0.8, ls='--')
    ax.set_xlabel('Tier 1 ko/Mb (resistance genes)')
    ax.set_ylabel('Spearman rho (detection freq vs Cr concentration)')
    ax.set_title('Cr dose-response rho vs Tier 1 density\n(colour = Tier 2 density)')
    plt.tight_layout()
    for ext in ['png', 'pdf']:
        plt.savefig(f'{FIGDIR}/fig10_dose_response.{ext}', dpi=180, bbox_inches='tight')
    plt.show()
    print("Saved fig10_dose_response")
else:
    print("No Cr data for plotting")


Genus presence matrix: 933 genera x 745 samples


/tmp/ipykernel_201629/2645775293.py:43: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, p = stats.spearmanr(q_medians[:len(det_freqs)], det_freqs)


  Cu: 0 genera skipped (insufficient quartile samples)


/tmp/ipykernel_201629/2645775293.py:43: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, p = stats.spearmanr(q_medians[:len(det_freqs)], det_freqs)


  Cr: 0 genera skipped (insufficient quartile samples)


/tmp/ipykernel_201629/2645775293.py:43: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, p = stats.spearmanr(q_medians[:len(det_freqs)], det_freqs)


  As: 0 genera skipped (insufficient quartile samples)


/tmp/ipykernel_201629/2645775293.py:43: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, p = stats.spearmanr(q_medians[:len(det_freqs)], det_freqs)


  Zn: 0 genera skipped (insufficient quartile samples)

Dose-response genus-metal pairs: 3732
Significant (q_BH < 0.1): 0 genus-metal pairs

Dose-response rho ~ ko_per_mb Spearman:
metal       predictor       rho        p  n_genera
   Cu ko_per_mb_total  0.079480 0.098986       432
   Cu ko_per_mb_tier1  0.040642 0.399433       432
   Cu ko_per_mb_tier2  0.121805 0.011284       432
   Cr ko_per_mb_total  0.108257 0.024437       432
   Cr ko_per_mb_tier1  0.112378 0.019472       432
   Cr ko_per_mb_tier2  0.090880 0.059116       432
   As ko_per_mb_total -0.016473 0.732795       432
   As ko_per_mb_tier1  0.015852 0.742507       432
   As ko_per_mb_tier2 -0.056846 0.238376       432
   Zn ko_per_mb_total  0.093882 0.051182       432
   Zn ko_per_mb_tier1  0.084504 0.079356       432
   Zn ko_per_mb_tier2  0.096639 0.044698       432


Saved fig10_dose_response


## Block 3: Tier 1 vs Tier 2 Within-MAG Co-occurrence

Partial Spearman rho (controlling for genome size) between Tier 1 and Tier 2 ko/Mb across all MAGs. Permutation-based p-value (n=1000 shuffles). Tests whether tiers are biologically distinct (rho~0) or redundant (rho>0.3).

In [4]:
print("Loading MAG KO density for tier co-occurrence...")
mag = pd.read_csv(f'{BASE}/data/mgnify_mag_ko_density.csv')

if mag['genus'].str.startswith('g__').any():
    mag['genus_lower'] = mag['genus'].str[3:].str.lower()
else:
    mag['genus_lower'] = mag['genus'].str.lower()

mag = mag[mag['genome_length_mb'] > 0.5].copy()
print(f"MAGs after completeness filter (>0.5 Mb): {len(mag)}")

if 'lineage' in mag.columns:
    mag['domain'] = mag['lineage'].str.split(';').str[0].str.strip()
else:
    mag['domain'] = 'All'

def residualize(x, cov):
    """OLS residuals: x ~ 1 + cov."""
    A = np.column_stack([np.ones(len(cov)), cov])
    b, _, _, _ = np.linalg.lstsq(A, x, rcond=None)
    return x - A @ b

def pearson_fast(x, y):
    """Fast Pearson r (no scipy overhead)."""
    xc = x - x.mean()
    yc = y - y.mean()
    denom = np.sqrt((xc**2).sum() * (yc**2).sum())
    return 0.0 if denom == 0 else np.dot(xc, yc) / denom

def tier_cooccurrence(df, label):
    t1 = df['ko_per_mb_tier1'].values.astype(float)
    t2 = df['ko_per_mb_tier2'].values.astype(float)
    sz = np.log1p(df['genome_length_mb'].values)
    n  = len(df)

    # Raw Spearman
    rho_raw, p_raw = stats.spearmanr(t1, t2)

    # Partial Spearman: residualise on log(genome_size), then rank-based Pearson
    r1 = residualize(t1, sz)
    r2 = residualize(t2, sz)
    r1r = rankdata(r1).astype(float)
    r2r = rankdata(r2).astype(float)
    rho_partial = pearson_fast(r1r, r2r)

    # Permutation p-value (fast: permute ranks, use pearson_fast)
    null_rhos = np.empty(N_PERM)
    perm_r1r  = r1r.copy()
    for i in range(N_PERM):
        np.random.shuffle(perm_r1r)
        null_rhos[i] = pearson_fast(perm_r1r, r2r)
    if rho_partial >= 0:
        p_perm = (null_rhos >= rho_partial).mean()
    else:
        p_perm = (null_rhos <= rho_partial).mean()
    p_perm = max(float(p_perm), 1.0 / N_PERM)

    # Bootstrap 95% CI
    boot_rhos = np.empty(N_BOOT)
    idx_pool  = np.arange(n)
    for i in range(N_BOOT):
        idx = np.random.choice(idx_pool, size=n, replace=True)
        boot_rhos[i] = pearson_fast(r1r[idx], r2r[idx])
    ci_lo, ci_hi = np.percentile(boot_rhos, [2.5, 97.5])

    return dict(domain=label, n_mags=n,
                rho_raw=round(rho_raw, 5), p_raw=round(p_raw, 6),
                rho_partial=round(rho_partial, 5), p_perm=round(p_perm, 6),
                CI_lo=round(ci_lo, 5), CI_hi=round(ci_hi, 5))

# Run for all MAGs, then by domain
results = [tier_cooccurrence(mag, 'All')]
domain_counts = mag['domain'].value_counts()
for dom in domain_counts.index:
    if domain_counts[dom] >= 1000:
        sub = mag[mag['domain'] == dom]
        print(f"  Running for {dom} (n={len(sub)})...")
        results.append(tier_cooccurrence(sub, dom))

tc = pd.DataFrame(results)
tc.to_csv(f'{BASE}/data/tier_cooccurrence.csv', index=False)
print("\nTier co-occurrence results:")
print(tc[['domain', 'n_mags', 'rho_raw', 'rho_partial', 'p_perm', 'CI_lo', 'CI_hi']].to_string(index=False))

# ---- Figure 11: 2D density hexbin ----
# Identify the most common domain label (for second panel)
top_domains = [d for d in domain_counts.index if domain_counts[d] >= 1000]
fig, axes = plt.subplots(1, min(2, 1 + len(top_domains)), figsize=(6 * min(2, 1 + len(top_domains)), 4))
if not hasattr(axes, '__len__'):
    axes = [axes]

panels = [('All', mag)] + [(d, mag[mag['domain'] == d]) for d in top_domains[:1]]
for ax, (label, sub) in zip(axes, panels):
    hb = ax.hexbin(sub['ko_per_mb_tier1'], sub['ko_per_mb_tier2'],
                   gridsize=60, cmap='YlOrRd', mincnt=1, bins='log')
    plt.colorbar(hb, ax=ax, label='log10(count)')
    row = tc[tc['domain'] == label]
    if len(row):
        rp = row.iloc[0]['rho_partial']
        pp = row.iloc[0]['p_perm']
        ax.set_title(f'{label}\nrho_partial={rp:.3f}, p_perm={pp:.4f}')
    ax.set_xlabel('Tier 1 ko/Mb')
    ax.set_ylabel('Tier 2 ko/Mb')

plt.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(f'{FIGDIR}/fig11_tier_cooccurrence.{ext}', dpi=180, bbox_inches='tight')
plt.show()
print("Saved fig11_tier_cooccurrence")


Loading MAG KO density for tier co-occurrence...


MAGs after completeness filter (>0.5 Mb): 260606


  Running for d__Bacteria (n=258420)...


  Running for d__Archaea (n=2186)...

Tier co-occurrence results:
     domain  n_mags  rho_raw  rho_partial  p_perm   CI_lo   CI_hi
        All  260606  0.99001      0.99935   0.001 0.99928 0.99941
d__Bacteria  258420  0.99196      0.99943   0.001 0.99937 0.99948
 d__Archaea    2186  0.87909      0.94606   0.001 0.93636 0.95449


Saved fig11_tier_cooccurrence


## Block 4: Hotspot Genus Identity — Tier 1 vs Tier 2 Split

Categorical: Mann-Whitney U comparing ko/Mb in hotspot-enriched (>0.2) vs background (<0.05) genera. Continuous sensitivity: Spearman rho(hotspot_frac, Tier 1/2 ko/Mb).

In [5]:
# Note: AlphaEarth uses GTDB accessions; MGnify MAG KO uses MGYG IDs.
# Join at genus level instead of genome_id.
print("Loading AlphaEarth hotspot data...")
ae = pd.read_csv(f'{BASE}/data/alphaearth_hotspot_comparison.csv')
print(f"AlphaEarth MAGs: {len(ae)}")

# Normalise genus names (GTDB 'g__Genus' format)
ae['genus_lower'] = ae['genus'].str.replace(r'^g__', '', regex=True).str.lower()

# Per-genus hotspot fraction from AlphaEarth (require >= 10 MAGs per genus)
ae_genus = ae.groupby('genus_lower').agg(
    hotspot_frac = ('is_hotspot', 'mean'),
    n_ae_mags    = ('genome_id', 'count'),
).reset_index()
ae_genus = ae_genus[ae_genus['n_ae_mags'] >= 10]
print(f"Genera with >= 10 AlphaEarth MAGs: {len(ae_genus)}")

# Load MAG KO density fresh (independent of Block 3); aggregate to genus
mag_ko2 = pd.read_csv(f'{BASE}/data/mgnify_mag_ko_density.csv')
if mag_ko2['genus'].str.startswith('g__').any():
    mag_ko2['genus_lower'] = mag_ko2['genus'].str[3:].str.lower()
else:
    mag_ko2['genus_lower'] = mag_ko2['genus'].str.lower()

mag_genus = mag_ko2.groupby('genus_lower')[
    ['ko_per_mb_total', 'ko_per_mb_tier1', 'ko_per_mb_tier2']
].mean().reset_index()

# Join AlphaEarth genus hotspot_frac with MGnify genus KO density
ae_ko = ae_genus.merge(mag_genus, on='genus_lower', how='inner')
print(f"Joined on genus_lower (AlphaEarth x MGnify): {len(ae_ko)} genera")

# ae_ko is already at genus level; rename n_ae_mags for clarity
g_hot = ae_ko.rename(columns={'n_ae_mags': 'n_mags'})
print(f"Genera in joined dataset: {len(g_hot)}")
g_hot.to_csv(f'{BASE}/data/hotspot_tier_split.csv', index=False)

# ---- Categorical: hotspot-enriched vs background (Mann-Whitney U) ----
enr = g_hot[g_hot['hotspot_frac'] > 0.2]
bg  = g_hot[g_hot['hotspot_frac'] < 0.05]
print(f"\nHotspot-enriched genera (>0.2): {len(enr)}")
print(f"Background genera (<0.05): {len(bg)}")

result_rows = []
for metric in ['ko_per_mb_total', 'ko_per_mb_tier1', 'ko_per_mb_tier2']:
    a, b = enr[metric].dropna().values, bg[metric].dropna().values
    if len(a) == 0 or len(b) == 0:
        continue
    U, p = stats.mannwhitneyu(a, b, alternative='two-sided')
    cliff_d = 1 - 2 * U / (len(a) * len(b))
    result_rows.append({'metric': metric, 'test': 'Mann-Whitney',
                        'U': U, 'p_value': p, 'cliff_delta': cliff_d,
                        'n_hotspot': len(a), 'n_background': len(b),
                        'rho': np.nan, 'n_genera': np.nan})

# ---- Continuous sensitivity: Spearman rho(hotspot_frac, Tier 1/2) ----
for metric in ['ko_per_mb_total', 'ko_per_mb_tier1', 'ko_per_mb_tier2']:
    v = g_hot[['hotspot_frac', metric]].dropna()
    rho, p = stats.spearmanr(v['hotspot_frac'], v[metric])
    result_rows.append({'metric': metric, 'test': 'Spearman_continuous',
                        'U': np.nan, 'p_value': p, 'cliff_delta': np.nan,
                        'n_hotspot': np.nan, 'n_background': np.nan,
                        'rho': rho, 'n_genera': len(v)})

hotspot_res = pd.DataFrame(result_rows)
hotspot_res.to_csv(f'{BASE}/data/hotspot_tier_mwu.csv', index=False)

print("\nMann-Whitney (hotspot-enriched vs background):")
mw = hotspot_res[hotspot_res['test'] == 'Mann-Whitney']
print(mw[['metric', 'cliff_delta', 'p_value', 'n_hotspot', 'n_background']].to_string(index=False))

print("\nContinuous Spearman rho(hotspot_frac, tier ko/Mb):")
cs = hotspot_res[hotspot_res['test'] == 'Spearman_continuous']
print(cs[['metric', 'rho', 'p_value', 'n_genera']].to_string(index=False))

# ---- Figure 12: boxplots ----
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, metric, label in zip(axes, ['ko_per_mb_tier1', 'ko_per_mb_tier2'], ['Tier 1', 'Tier 2']):
    a, b = enr[metric].dropna().values, bg[metric].dropna().values
    if len(a) == 0 or len(b) == 0:
        ax.set_visible(False)
        continue
    bp = ax.boxplot([a, b], labels=['Hotspot\n(>0.2)', 'Background\n(<0.05)'],
                    patch_artist=True, medianprops={'color': 'black', 'linewidth': 2})
    bp['boxes'][0].set_facecolor('#e74c3c')
    bp['boxes'][1].set_facecolor('#95a5a6')
    row = mw[mw['metric'] == metric]
    p_v = row.iloc[0]['p_value'] if len(row) else 1.0
    ax.set_title(f'{label} ko/Mb\np(MWU)={p_v:.4f}')
    ax.set_ylabel('ko per Mb')

plt.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(f'{FIGDIR}/fig12_hotspot_tier.{ext}', dpi=180, bbox_inches='tight')
plt.show()
print("Saved fig12_hotspot_tier")


Loading AlphaEarth hotspot data...
AlphaEarth MAGs: 36971
Genera with >= 10 AlphaEarth MAGs: 608


Joined on genus_lower (AlphaEarth x MGnify): 394 genera
Genera in joined dataset: 394

Hotspot-enriched genera (>0.2): 71
Background genera (<0.05): 231

Mann-Whitney (hotspot-enriched vs background):
         metric  cliff_delta  p_value  n_hotspot  n_background
ko_per_mb_total     0.079142 0.313569       71.0         231.0
ko_per_mb_tier1     0.081458 0.299602       71.0         231.0
ko_per_mb_tier2     0.056765 0.469919       71.0         231.0

Continuous Spearman rho(hotspot_frac, tier ko/Mb):
         metric       rho  p_value  n_genera
ko_per_mb_total -0.001360 0.978535     394.0
ko_per_mb_tier1  0.010916 0.828989     394.0
ko_per_mb_tier2 -0.000160 0.997482     394.0


/tmp/ipykernel_201629/2426390852.py:83: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot([a, b], labels=['Hotspot\n(>0.2)', 'Background\n(<0.05)'],
/tmp/ipykernel_201629/2426390852.py:83: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot([a, b], labels=['Hotspot\n(>0.2)', 'Background\n(<0.05)'],


Saved fig12_hotspot_tier


## Block 5: Synthesis — Experimental Design Signals

Consolidates significant results across all four analyses into a design table.

In [6]:
rows = []

# CWM significant results
cwm = pd.read_csv(f'{BASE}/data/cwm_ngsa_spearman.csv')
for _, r in cwm[cwm['q_BH'] < 0.1].iterrows():
    tier = r['cwm_type'].replace('CWM_', '')
    rows.append({
        'analysis':      'CWM ~ NGSA metal',
        'metric':         f"{r['cwm_type']} ~ {r['metal']}",
        'tier':            tier,
        'rho_or_effect':   round(r['rho'], 3),
        'p_value':         r['p'],
        'q_BH':            r['q_BH'],
        'n':               r['n_samples'],
        'implication': ('Community gene investment rises with metal conc'
                        if r['rho'] > 0 else 'Community gene investment falls with metal conc'),
    })

# Dose-response significant results (p < 0.05)
dr = pd.read_csv(f'{BASE}/data/dose_response_pgls.csv')
for _, r in dr[dr['p'] < 0.05].iterrows():
    rows.append({
        'analysis':      'Dose-response rho ~ ko_per_mb',
        'metric':         f"{r['metal']} ~ {r['predictor']}",
        'tier':            r['predictor'].replace('ko_per_mb_', ''),
        'rho_or_effect':   round(r['rho'], 3),
        'p_value':         r['p'],
        'q_BH':            np.nan,
        'n':               r['n_genera'],
        'implication': ('High-KO genera enrich under metal stress' if r['rho'] > 0
                        else 'High-KO genera decline under metal stress'),
    })

# Tier co-occurrence
tc = pd.read_csv(f'{BASE}/data/tier_cooccurrence.csv')
a = tc[tc['domain'] == 'All'].iloc[0]
rows.append({
    'analysis':      'Tier 1 vs Tier 2 co-occurrence',
    'metric':         f"Partial Spearman (n={int(a['n_mags'])} MAGs)",
    'tier':           'both',
    'rho_or_effect':   round(a['rho_partial'], 3),
    'p_value':         a['p_perm'],
    'q_BH':            np.nan,
    'n':               int(a['n_mags']),
    'implication': ('Tiers co-occur: treat as unified predictor' if a['rho_partial'] > 0.3
                    else 'Tiers independent: measure separately in microcosm'),
})

# Hotspot continuous Spearman
ht = pd.read_csv(f'{BASE}/data/hotspot_tier_mwu.csv')
for _, r in ht[ht['test'] == 'Spearman_continuous'].iterrows():
    rows.append({
        'analysis':      'Hotspot occupancy (continuous Spearman)',
        'metric':         f"hotspot_frac ~ {r['metric']}",
        'tier':            r['metric'].replace('ko_per_mb_', ''),
        'rho_or_effect':   round(float(r['rho']), 3),
        'p_value':         r['p_value'],
        'q_BH':            np.nan,
        'n':               r.get('n_genera', np.nan),
        'implication': ('Gene-dense genera geographically clustered' if float(r['rho']) > 0
                        else 'Gene-dense genera cosmopolitan, not hotspot-concentrated'),
    })

synth = pd.DataFrame(rows)
synth.to_csv(f'{BASE}/data/experimental_design_signals.csv', index=False)
print("=== NB19 SIGNALS BEYOND NICHE BREADTH: SYNTHESIS ===\n")
print(synth[['analysis', 'metric', 'tier', 'rho_or_effect', 'p_value', 'implication']].to_string(
    index=False, max_colwidth=55))


=== NB19 SIGNALS BEYOND NICHE BREADTH: SYNTHESIS ===

                               analysis                           metric  tier  rho_or_effect      p_value                                             implication
                       CWM ~ NGSA metal                   CWM_total ~ Ni total          0.182 5.940436e-07         Community gene investment rises with metal conc
                       CWM ~ NGSA metal                   CWM_tier1 ~ Co tier1          0.186 3.298146e-07         Community gene investment rises with metal conc
                       CWM ~ NGSA metal                   CWM_tier1 ~ Ni tier1          0.177 1.189285e-06         Community gene investment rises with metal conc
                       CWM ~ NGSA metal                   CWM_total ~ Cu total          0.173 2.540677e-06         Community gene investment rises with metal conc
                       CWM ~ NGSA metal                   CWM_tier2 ~ Pb tier2         -0.168 4.097118e-06         Community gene i